# 07 — the dark bound, and the stack for `eta_comb`

Session 03, capturing to `protocols/03-dark-bound.md`. No light source, cap on,
about five hours at gain 250, offset 15, -10 C.

**What this notebook is for.** Capturing one night of darks and interleaved bias,
and measuring three things from them: an upper bound on dark current `D` at -10 C;
an upper bound on the combination efficiency `eta_comb` from a stack that needs no
registration (L15); and whether the dark carries spatial structure — glow, or
dark-signal non-uniformity — inside the ROI the model will use.

**What it is not for.** It is not a `D(T)` sweep: MISSION fixes the setpoint at
-10 C, so temperature is not an axis here and no doubling temperature is fitted
(L14 is explicit that there is nothing to fit one to). It does not integrate
anything — stacking is a PixInsight step and belongs to build step 5; tonight
captures. And it does not touch the archive: every frame here is shot to a
protocol, so its header is trusted in full.

**The result this is designed to produce is a bound.** The best outcome is that
the slope does not exceed its own uncertainty, `D` leaves `sigma^2` in the model,
and the temperature axis never opens. An upper bound is a result and is recorded
as one.

**The error bar is the pedestal, not the frame count.** Rule 2 of the protocol:
the statistical floor is 1.3e-6 e-/px/s at 600 s, four decades below the number
being tested, so nothing here is limited by how many frames it shoots. What limits
it is pedestal wander — 11.7 counts across a bracket would fake `D = 0.01 e-/px/s`.
That is why the bias blocks are interleaved on a 20-minute clock rather than
batched, and why they are a published result in their own right.

The explaining half is `08`.

## The plan, before the camera is opened

Every number here is measured, not fitted: gain 250 is a swept point in both
session 01 and session 02, which is why the protocol moved off 252 (session 02's
gain law has a 1.34% residual against its own 1% rule, so `g` is not interpolable).

In [ ]:
import datetime as dt
import json
import pathlib
import sys
import time

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import asi, fits as F, spatial as SP, stats as ST

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session03"
FRAMES = DATA / "frames"
FRAMES.mkdir(parents=True, exist_ok=True)

GAIN = 250              # a swept point in sessions 01 and 02; 252 is not
OFFSET = 15             # results/bias_constants.json, project_offset
ROI = (1408, 568, 1024, 1024)          # as session 01
FULL_FRAME = (0, 0, 3840, 2160)
SETPOINT_C = asi.SETPOINT_C

FRAME_GAP_S = 0.2       # readout, USB and the file write are what heat the sensor
HOLD_TIMEOUT_S = 600.0  # a hold that never ends is a cooler fault, not a wait
MAX_RETAKES = 3         # a retaken 600 s dark costs ten minutes; three is the budget

# Session 01 and 02, at exactly this gain and offset.  Predictions for the
# pre-flight to check against, never inputs to anything published tonight.
bias_c = json.loads((RESULTS / "bias_constants.json").read_text())
ptc_c = json.loads((RESULTS / "ptc_constants.json").read_text())
G_E_PER_COUNT = ptc_c["system_gain"]["value"]["250"]
G_ERR = ptc_c["system_gain"]["uncertainty"]["250"]
PEDESTAL_PRED = 76.66   # results/bias_sweep.csv, gain 250 at offset 15
R_COUNTS = 1.7281       # ditto, R_sd

print(f"gain {GAIN}, offset {OFFSET}, ROI {ROI}, setpoint {SETPOINT_C} C")
print(f"g = {G_E_PER_COUNT} +/- {G_ERR} e-/count   "
      f"(one count is {G_E_PER_COUNT:.3f} e-, which is the quantisation L14 lost to)")
print(f"predicted pedestal {PEDESTAL_PRED} counts, R {R_COUNTS} counts")

plane_px = ROI[2] * ROI[3] // 4
stat_floor = np.sqrt(2) * R_COUNTS / np.sqrt(plane_px * 10) * G_E_PER_COUNT / 600
print(f"\nstatistical floor on D at 600 s: {stat_floor:.2e} e-/px/s")
print(f"pedestal wander that would fake D = 0.01 e-/px/s: "
      f"{0.01 * 600 / G_E_PER_COUNT:.1f} counts")
print(f"                          ... and 1e-4 e-/px/s: "
      f"{1e-4 * 600 / G_E_PER_COUNT:.2f} counts")

## The schedule

One list of blocks, in execution order. The interleave is on a 20-minute clock:
every dark block is bracketed by a bias block before and after, and no bracket
spans more than 20 minutes of wall clock. The block index is in the filename, so
the time order survives on disk even if a header is ever in doubt — but `DATE-OBS`
is what the analysis interpolates on, because that is the actual clock.

The last interleaved bias block of D4 *is* the protocol's B4; there is no separate
trailing block.

In [ ]:
N_BIAS = 10             # per block; sets the pedestal reference to 1.1e-3 counts
N_BIAS_FF = 5           # a full frame has 8x the pixels, so 5 beats the ROI's 10
BIAS = None             # resolved from the camera as its minimum exposure


def schedule(bias_s):
    """The night, as `(kind, exposure_s, n, roi)` blocks in execution order."""
    b = ("bias", bias_s, N_BIAS, ROI)
    ff_b = ("bias", bias_s, N_BIAS_FF, FULL_FRAME)
    blocks = [b,
              ("dark", 1.0, 20, ROI), b,        # D1
              ("dark", 60.0, 20, ROI), b]       # D2, one 20-minute bracket
    for _ in range(8):                          # D3: 32 darks at 300 s
        blocks += [("dark", 300.0, 4, ROI), b]
    for _ in range(4):                          # D4: 8 darks at 600 s
        blocks += [("dark", 600.0, 2, ROI), b]  # the last b is B4
    # The full-frame block is 40 minutes long and used to sit unbracketed at the
    # end of the night.  The 2026-08-31 smoke test measured the pedestal moving
    # 1.1 counts in ten minutes, so 40 minutes unbracketed is worth several
    # counts -- against the 0.12 that would fake D = 1e-4 e-/px/s.  It gets its
    # own bias either side, at its own ROI.
    blocks += [ff_b, ("dark", 600.0, 4, FULL_FRAME), ff_b]
    return blocks


plan = schedule(3.2e-05)          # the published minimum, for planning only
frames = sum(n for _, _, n, _ in plan)
seconds = sum(n * (e + FRAME_GAP_S + 0.4) for _, e, n, _ in plan)
gb = sum(n * r[2] * r[3] * 2 for _, _, n, r in plan) / 1e9

print(f"{len(plan)} blocks, {frames} frames, {gb:.2f} GB, "
      f"{seconds / 3600:.2f} h of capture")
for kind, e, n, r in plan:
    print(f"  {kind:5s} {e:7.4g} s x {n:2d}  {r[2]}x{r[3]}")

## Gate 1 — white balance, proved in the pixels

Nothing captured before this passes is usable (L01). Reading the control back only
proves the control took; the evidence is a modal step of 16 on all four planes.
Run on stored values, because the test goes vacuous in ADC counts.

In [ ]:
existing = list(FRAMES.glob("*.fits"))
if existing:
    print(f"WARNING: {len(existing)} frames already in {FRAMES}.\n"
          "A resumed night has a gap in the pedestal series.  That is recoverable\n"
          "— DATE-OBS says where — but the blocks either side of the gap are two\n"
          "brackets, not one, and 08 must treat them that way.")

rig = asi.open_camera()
print("gain range     ", rig.range("Gain"))
print("exposure range ", rig.range("Exposure"), "us")
print("white balance shipped:", rig.get("WB_R"), rig.get("WB_B"))

asi.neutralise_white_balance(rig)
asi.set_roi(rig, *ROI)
asi.configure(rig, gain=GAIN, offset=OFFSET)
BIAS = rig.min_exposure_s()               # measured, never assumed

for _ in range(2):                        # discards after the configuration change
    asi.capture(rig, BIAS)

gate1 = {}
for _ in range(5):
    mosaic, _ = asi.capture(rig, BIAS)
    for name, plane in SP.split(mosaic).items():     # stored values: the grid is 16 there
        gate1.setdefault(name, []).append(ST.value_step(plane))

for name in SP.PLANES:
    print(f"  {name:2s} modal step {gate1[name]}")
bad = {n: s for n, s in gate1.items() if set(s) != {16}}
assert not bad, (f"white balance is still being applied: {bad} -- stop the session, "
                 "nothing captured from here is usable (L01)")
print(f"\ngate 1 passed on five frames.  bias exposure {BIAS * 1e6:.0f} us")

## Gate 2 — the pedestal is where session 01 left it

A cheap check with a real failure behind it: if the pedestal at gain 250 does not
land near session 01's 76.66 counts, then either the gain or the offset is not what
this notebook thinks, and every subtraction tonight is against the wrong level.
Five counts is a wide band — this catches a wrong *setting*, not a drift.

In [ ]:
mosaic, hdr = asi.capture(rig, BIAS, imagetyp="BIAS")
levels = {n: float(ST.to_adc(p).mean()) for n, p in SP.split(mosaic).items()}
mean_level = float(np.mean(list(levels.values())))

for n, v in levels.items():
    print(f"  {n:2s} {v:8.3f} counts")
print(f"\nmean {mean_level:.3f} vs session 01's {PEDESTAL_PRED} "
      f"({mean_level - PEDESTAL_PRED:+.3f})")
print(f"header says gain {hdr['GAIN']}, offset {hdr['OFFSET']}")
assert abs(mean_level - PEDESTAL_PRED) < 5.0, (
    f"pedestal {mean_level:.2f} is not session 01's {PEDESTAL_PRED} at gain "
    f"{GAIN}/offset {OFFSET} -- check the settings before capturing anything")
assert hdr["GAIN"] == GAIN and hdr["OFFSET"] == OFFSET
print("gate 2 passed")

## Cool down, and log the curve

Ambient is assumed 25 C. Session 02 cooled from exactly 25.0 C and held -10 C on
65% duty with zero holds and zero retakes; tonight reads the sensor far less often,
and readout is what heats it, so the load is lighter than a run that already passed.
The duty at setpoint is the number to watch — it is the headroom this room leaves.

In [ ]:
AMBIENT_START_C = 25.0            # <- record the real reading before running

# Appended for the same reason `frames.csv` is: a resumed night has two
# cool-downs, and the first one is the one that describes the room.
cool_path = DATA / "cooldown.csv"
fresh = not cool_path.exists()
cool_log = open(cool_path, "a", newline="")
if fresh:
    cool_log.write("elapsed_s,temp_C,duty_pct\n")


def show(elapsed, temp, duty):
    cool_log.write(f"{elapsed},{temp},{duty}\n")
    cool_log.flush()                     # the reading exists nowhere else
    if int(elapsed) % 30 == 0:
        print(f"  {elapsed:6.0f} s  {temp!s:>6} C  {duty:>3}%", flush=True)


try:
    trace = asi.cool_to(rig, SETPOINT_C, log=show)
finally:
    cool_log.close()

temps = [t for _, t, _ in trace if t is not None]
duty = trace[-1][2]
print(f"\nsettled in {trace[-1][0]:.0f} s;  {temps[0]} C -> {temps[-1]} C")
print(f"duty at setpoint {duty}%")
if duty > 85:
    print("  ^ under 15% headroom.  Five hours is a long time to hold that; "
          "check the fan and the ambient before starting.")

## Capture

The library does one frame; the loop is here (`CLAUDE.md`). Two rules the loop
enforces, both from the protocol:

- **Every frame outside the band is retaken rather than written.** A retaken 600 s
  dark costs ten minutes, so the budget is three; past that the block is recorded
  as it stands and the night is judged in `08` rather than silently patched.
- **Nothing changes between blocks except exposure.** Gain, offset and setpoint are
  set once, above. The ROI changes exactly once, for the full-frame block, after
  the ROI run has closed with its own bias block.

In [ ]:
log_path = DATA / "capture_log.txt"
log_file = open(log_path, "a", encoding="utf8")


def say(msg):
    print(msg, flush=True)
    log_file.write(msg + "\n")
    log_file.flush()            # data/ is gitignored and this is the session record


def in_band(header):
    t = header["CCD-TEMP"]
    return t is not None and abs(t - SETPOINT_C) <= asi.BAND_C


def capture_block(index, kind, exposure_s, n, roi):
    """One block of `n` frames at one exposure, written in time order.

    Returns the rows the session record needs.  A frame out of band is retaken;
    a frame that exhausts `MAX_RETAKES` is written anyway and flagged, because
    a gap in the series is worse than a frame `08` can exclude by its header.

    The cooler duty is recorded per frame, and it is not decoration.  The
    2026-08-31 smoke test held -10.0 C exactly while the duty climbed 64 -> 77%
    over half an hour and the pedestal moved 1.1 counts: the sensor was cold
    and the body was still equilibrating.  Sensor temperature alone cannot see
    that, so if the pedestal series turns out to track anything tonight, duty
    is the column `08` needs to test it against.  Recording it costs one USB
    read per frame and turns a confound into a measurement.
    """
    rows, retakes = [], 0
    for i in range(n):
        path = FRAMES / f"blk{index:02d}_{kind}_{i:03d}.fits"
        if path.exists():
            continue                                   # resuming; see the warning above
        for attempt in range(MAX_RETAKES + 1):
            mosaic, header = asi.capture(rig, exposure_s, imagetyp=kind.upper())
            if in_band(header) or attempt == MAX_RETAKES:
                break
            retakes += 1
        header["BLOCK"] = index
        duty = rig.get("CoolPowerPerc")
        F.write(path, mosaic, header)
        rows.append({"block": index, "kind": kind, "i": i, "file": path.name,
                     "exptime": header["EXPTIME"], "ccd_temp": header["CCD-TEMP"],
                     "duty_pct": duty, "roi_w": roi[2],
                     "date_obs": header["DATE-OBS"], "in_band": in_band(header)})
        time.sleep(FRAME_GAP_S)
    return rows, retakes


say(f"\n===== session 03 starting {dt.datetime.now():%Y-%m-%d %H:%M} =====")
say(f"gain {GAIN}, offset {OFFSET}, setpoint {SETPOINT_C} C, ambient "
    f"{AMBIENT_START_C} C, bias exposure {BIAS * 1e6:.0f} us")

In [ ]:
import csv

plan = schedule(BIAS)
t0, all_rows, total_retakes = time.monotonic(), [], 0
current_roi = ROI

for index, (kind, exposure_s, n, roi) in enumerate(plan):
    if roi != current_roi:
        say(f"  ROI -> {roi}")
        asi.set_roi(rig, *roi)
        asi.configure(rig, gain=GAIN, offset=OFFSET)   # re-asserted after an ROI change
        asi.capture(rig, exposure_s if exposure_s < 5 else 1.0)   # discard
        current_roi = roi

    started = time.monotonic()
    rows, retakes = capture_block(index, kind, exposure_s, n, roi)
    all_rows += rows
    total_retakes += retakes

    temps = [r["ccd_temp"] for r in rows if r["ccd_temp"] is not None]
    span = f"{min(temps)} to {max(temps)} C" if temps else "no reading"
    say(f"blk{index:02d} {kind:5s} {exposure_s:7.4g} s x {n:2d}  "
        f"{(time.monotonic() - started) / 60:5.1f} min  {span}"
        f"{f'  {retakes} retaken' if retakes else ''}"
        f"   [{(time.monotonic() - t0) / 3600:.2f} h elapsed]")

# Appended, not rewritten.  On a resumed night the rows from before the
# interruption are already in this file and describe frames that cannot be
# recaptured; opening it "w" would delete the record of half the session while
# leaving the frames themselves on disk.  The header goes in only when new.
FIELDS = ["block", "kind", "i", "file", "exptime", "ccd_temp", "duty_pct",
          "roi_w", "date_obs", "in_band"]
csv_path = DATA / "frames.csv"
fresh = not csv_path.exists()
with open(csv_path, "a", newline="", encoding="utf8") as fh:
    w = csv.DictWriter(fh, fieldnames=FIELDS)
    if fresh:
        w.writeheader()
    w.writerows(all_rows)

say(f"\n{len(all_rows)} frames written in {(time.monotonic() - t0) / 3600:.2f} h, "
    f"{total_retakes} retaken, {sum(not r['in_band'] for r in all_rows)} out of band")
if not all_rows:
    say("nothing was captured -- every block was already on disk")
log_file.close()

In [ ]:
AMBIENT_END_C = None      # <- record the real reading, then close the camera

print(f"ambient {AMBIENT_START_C} C -> {AMBIENT_END_C} C")
rig.close()
print("camera closed; the TEC is off and the sensor is warming")

## The analysis half

Written after the frames land, against the analysis rules already fixed in
`protocols/03-dark-bound.md` — the rules are pre-registered, the code is not
written blind. It publishes to `results/`:

| file | what |
|---|---|
| `dark_blocks.csv` | one row per block: mean per plane, exposure, time, temperature |
| `pedestal_series.csv` | the ~15 interleaved bias blocks against wall clock |
| `dark_constants.json` | `D` as a bound, the pedestal stability that bounded it, DSNU, `eta_comb` |